# Data Processing - iFood Challenge

In [0]:
print(spark)
spark.range(5).show()

In [0]:
display(dbutils.fs.ls("/"))

In [0]:
display(dbutils.fs.ls("/Workspace/Users/jekasores@gmail.com"))

## 1) Import libraries

In [0]:
import sys
sys.path.append('/Workspace/Users/jekasores@gmail.com/ifood-case/src')

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import to_date, date_format
from pyspark.ml.feature import StringIndexer
from visualization_functions import *
from pyspark.sql.window import Window

## 2) Read datasets

In [0]:
# Path definition
base_path = "dbfs:/Workspace/Users/jekasores@gmail.com/ifood-case/data/raw"   # exemplo comum no Databricks
offers_path = f"{base_path}/offers.json"
profile_path = f"{base_path}/profile.json"
transactions_path = f"{base_path}/transactions.json"

# Read json files
offers_df = spark.read.option("multiLine", True).json(offers_path)
profiles_df = spark.read.option("multiLine", True).json(profile_path)
transactions_df = spark.read.option("multiLine", True).json(transactions_path)

# Display schema and first rows of the dataframes
print("=== offers schema ===")
offers_df.printSchema()
offers_df.show(5, truncate=False)

print("=== profiles schema ===")
profiles_df.printSchema()
profiles_df.show(5, truncate=False)

print("=== transactions schema ===")
transactions_df.printSchema()
transactions_df.show(10, truncate=False)


**Preprocessing ideas:**

*offers table*
- Separate channels that are saved into lists into dummy columns and drop channels column

*profiles*
- create a cohort columns using registered_on with YYYY-MM and drop registered_on column

*transactions*
- Separate schema from column value into 4 columns. Teorically, it represents: offer_id, reward, transaction amount. However, this schema has 4 positions instead of 3, some investigation needs to be done towards that.

**EDA ideas:**

*offers table*

- Check discount range
- Number of possible channels (categories)
- Min value to activate the offer range
- Number of possible offer types (categories)
- % of missing values per columns
- Number of unique offer id vs. number of rows in the dataframe

*profiles table*
- check cohort date
- check gender distribution (categorical)
- Check age distribution (histogram and boxplot)
- Check credit card limit distribution (histogram and boxplot)
- % of missing values per columns
- Number of unique profile ids vs. number of rows in the dataframe

*transactions table*
- check offer type distribution (categorical)

**Joining datasets**

- profile.id <--> transactions.account_id
- offers.id <--> transactions.value.offer_id

**Possible additional viz**
- Period between received → viewed → completed
- % of offers received that were viewed (CHECK POSSIBILITY)
- % of visualized offers that were completed (CHECK POSSIBILITY)
- Mean amount spent of completed offers vs. not accepted offers
- Correlation plot

**Possible additional fatures**
- Number of days between cohort and offer date (CHECK POSSIBILITY)

## 3) Preprocessing

### 3.1) Offers table

In [0]:
distinct_channels = [
    row.channel
    for row in offers_df
    .select(F.explode("channels").alias("channel"))
    .distinct()
    .collect()
]
distinct_channels

In [0]:
#Criar colunas dummies para cada canal
offers_with_dummies = offers_df

for ch in distinct_channels:
    col_name = f"channel_{ch}"
    offers_with_dummies = offers_with_dummies.withColumn(
        col_name,
        F.when(F.array_contains(F.col("channels"), ch), 1).otherwise(0)
    )

In [0]:
#Remover a coluna original de lista
offers_final = offers_with_dummies.drop("channels")

In [0]:
offers_final.printSchema()

In [0]:
offers_final.show(5, truncate=False)

In [0]:
offers_final.display()

### 3.2) Profiles table

In [0]:
# Convert registered_on_date into date
profiles_final = profiles_df.withColumn(
    "registered_on_date",
    to_date(F.col("registered_on"), "yyyyMMdd")
)

# Create cohort YYYY-MM column
profiles_final = profiles_final.withColumn(
    "cohort",
    date_format(F.col("registered_on_date"), "yyyy-MM")
)

# Indexing by gender
gender_indexer = StringIndexer(
    inputCol="gender",
    outputCol="gender_index",
    handleInvalid="keep"   # mantém valores nulos/unknown como categoria separada
)

gender_model = gender_indexer.fit(profiles_final)
profiles_final = gender_model.transform(profiles_final)

# Show mapped labels
print("Gender labels (ordem -> index):")
print(gender_model.labels)

profiles_final.show(5)
profiles_final.printSchema()

- Drop columns gender and registered_on for the model

### 3.3) Transactions table

In [0]:
# Transform elements inside value column in different collumns
transactions_final = transactions_df \
    .withColumn(
        "offer_id_from_struct",
        F.coalesce(
            F.col("value.offer_id"),      # alguns arquivos vêm assim
            F.col("value.offer id")       # outros vêm com espaço
        )
    ) \
    .withColumn("amount", F.col("value.amount")) \
    .withColumn("reward", F.col("value.reward"))

# Create final column offer_id
transactions_final = transactions_final.withColumn(
    "offer_id",
    F.when(F.col("event") == "transaction", F.lit(None))
     .otherwise(F.col("offer_id_from_struct"))
)

# Remove unused columns
transactions_final = transactions_final.drop("value", "offer_id_from_struct")


In [0]:
transactions_final.printSchema()
transactions_final.show(5)

### 3.4) Joining tables

In [0]:

# Renomear IDs para evitar colisão
profiles_dim = profiles_final.withColumnRenamed("id", "profile_id")
offers_dim = offers_final.withColumnRenamed("id", "offer_id_dim")
# Join transactions + profiles
tx_profiles = (
    transactions_final
    .join(
        profiles_dim,
        transactions_final.account_id == profiles_dim.profile_id,
        how="left"
    )
    .drop(profiles_dim.profile_id)
)

# Join com offers
joined_df = (
    tx_profiles
    .join(
        offers_dim,
        tx_profiles.offer_id == offers_dim.offer_id_dim,
        how="left"
    )
    .drop(offers_dim.offer_id_dim)
)

joined_df.printSchema()
joined_df.show(5, truncate=False)

In [0]:
joined_df.display()

In [0]:
joined_df = joined_df.drop("registered_on", "registered_on_date")
joined_df.display()

## 4) EDA

### 4.1) Offers table

In [0]:
offers_final.display()

In [0]:
#Dataframe shape
spark_shape(offers_final)

In [0]:
#Unique IDs analysis
result = check_unique_ids(offers_final, "id")
print(result)

In [0]:
plot_missing_data(offers_final)

In [0]:
plot_numerical_distributions(offers_final)

In [0]:
plot_numerical_boxplots(offers_final)

In [0]:
plot_categorical_distribution_same_style(offers_final, ["offer_type", "channel_mobile", "channel_email", "channel_social", "channel_web"])

In [0]:
offers_discount = offers_final.filter(F.col("offer_type") == "discount")
offers_bogo = offers_final.filter(F.col("offer_type") == "bogo")
offers_informational = offers_final.filter(F.col("offer_type") == "informational")

In [0]:
offers_discount.describe().display()

In [0]:
offers_bogo.describe().display()

In [0]:
offers_informational.describe().display()

**Offer types characteristics**
- *Discount:* $2 to $5 of discount value with duration of 7 to 10 days;
- *Bogo:* $5 to $10 of discount value with duration of 5 to 7 days;
- *Informational:* $0 of discount value with duration of 3-4 days.

### 4.2) Profiles table

In [0]:
profiles_final.display()

In [0]:
#Dataframe shape
spark_shape(profiles_final)

In [0]:
#Unique IDs analysis
result = check_unique_ids(profiles_final, "id")
print(result)

In [0]:
plot_missing_data(profiles_final)

In [0]:
plot_numerical_distributions(profiles_final)

In [0]:
plot_numerical_boxplots(profiles_final)

In [0]:
plot_categorical_distribution_same_style(profiles_final, ["cohort", "gender_index"], (14, 12))

In [0]:
profiles_final.select(
    F.min("cohort").alias("min_cohort"),
    F.max("cohort").alias("max_cohort")
).show()

### 4.3) Transactions table

In [0]:
transactions_final.display()

In [0]:
#Dataframe shape
spark_shape(transactions_final)

In [0]:
#Unique IDs analysis
result = check_unique_ids(transactions_final, "account_id")
print(result)

In [0]:
#Unique IDs analysis
result = check_unique_ids(transactions_final, "offer_id")
print(result)

In [0]:
def _compute_missing(df):
    """
    Computes missing values for both numerical and categorical columns.
    Missing definition:
    - Nulls
    - NaN
    - Strings with only whitespace
    """
    df_tmp = df

    # Categorical / object columns
    cat_cols = [
        field.name
        for field in df_tmp.schema.fields
        if field.dataType.typeName() in ["string"]
    ]

    # Numerical columns
    num_cols = [
        field.name
        for field in df_tmp.schema.fields
        if field.dataType.typeName() in ["double", "float", "integer", "long", "decimal"]
    ]

    from pyspark.sql import functions as F

    missing_cat = [
        F.sum(
            (
                F.col(col).isNull() | (F.trim(F.col(col)) == "")
            ).cast("int")
        ).alias(col)
        for col in cat_cols
    ]

    missing_num = [
        F.sum(
            (
                F.col(col).isNull() | F.isnan(F.col(col))
            ).cast("int")
        ).alias(col)
        for col in num_cols
    ]

    result = df_tmp.agg(
        *missing_cat,
        *missing_num
    )
    display(result)

In [0]:
_compute_missing(transactions_final)

In [0]:
plot_missing_data(transactions_final)

In [0]:
plot_numerical_distributions(transactions_final)

In [0]:
plot_numerical_boxplots(transactions_final)

In [0]:
plot_categorical_distribution_same_style(transactions_final, ["event"])

In [0]:
transaction_transacion = transactions_final.filter(F.col("event") == "transaction")
transaction_o_received = transactions_final.filter(F.col("event") == "offer received")
transaction_o_viewed = transactions_final.filter(F.col("event") == "offer viewed")
transaction_o_completed = transactions_final.filter(F.col("event") == "offer completed")


In [0]:
cols = ["time_since_test_start", "reward", "amount"]
plot_numerical_distributions(transaction_transacion.select(*cols))

In [0]:
plot_numerical_distributions(transaction_o_received.select(*cols))

In [0]:
plot_numerical_distributions(transaction_o_viewed.select(*cols))

In [0]:
plot_numerical_distributions(transaction_o_completed.select(*cols))

In [0]:
transactions_final.describe().display()

### 4.4) Joined table

In [0]:
joined_df.display()

**Assumptions**
- The only time columns are cohort(YYYY-MM from column registeredon from table profile) and time_since_test_start
(time since the beggining of the test in days).
- Given that, I believe this is an experiments of sending cupons for recently entered customers. For that, we have customers data of 5 years (2013-07 to 2018-07).
- There is a funnel when a customer do a transaction. First the customer does a transaction. Then, the customer receives the offer, then the customer views the offer; then the customer accepts the offer (main goal).
- It is possible to do a model that predicts the type of offer to send to each customer (using offer_type column as the target). However, talking about businnes it would be also intereting to predict:
    - the best channel to send the offer for each customer
- To train the model, filter only order that were completed using the target offer_type
- Each offer_type has specific characteristics of dicount value and duration days. Therefore, using offers table features in the final column would probably cause data leakage
- I count create a month feature. I could separate training, valudation, and test according to chronological order using cohort column.

**Assumptions - refined**
- *Some issues with the first assumption:*
  - Although it is technically possible to frame the problem as predicting the offer_type to be sent to each customer, this formulation does not properly reflect the core business objective. The offer type is already known at the moment of sending; the real decision problem is whether a given offer will be effective for a specific customer.
  - Using offer_type as the target variable would shift the problem to a multi-class classification that does not directly measure customer engagement or conversion, potentially resulting in a model that is not actionable from a marketing perspective.
  - Filtering the dataset to include only offer completed events removes the negative class (customers who received an offer but did not complete it).
- *Refined approach:*
  - The refined problem is framed as a binary classification task:
    - Given a customer, an offer, and the moment the offer is received, predict whether the customer will complete the offer within its validity period.
  - The target variable is defined as converted, where:
    - converted = 1 if the customer completes the offer within the duration specified by the offer configuration.
    - converted = 0 otherwise.
  - The dataset is built at the customer–offer exposure level, where each row represents a specific offer received by a customer at a given point in time.


## 5) Create final dataset

### 5.1) Create target

In [0]:
# Each row represents a customer receiving an offer at a specific time
offers_received = (
    transactions_final
    .filter(F.col("event") == "offer received")
    .select(
        F.col("account_id"),
        F.col("offer_id"),
        F.col("time_since_test_start").alias("t_received")
    )
)
offers_received.display()

In [0]:
# Offer completion events, keeping completion time
offers_completed = (
    transactions_final
    .filter(F.col("event") == "offer completed")
    .select(
        F.col("account_id"),
        F.col("offer_id"),
        F.col("time_since_test_start").alias("t_completed")
    )
)
offers_completed.display()

In [0]:
# Join received offers with completions
dataset = (
    offers_received
    .join(
        offers_completed,
        on=["account_id", "offer_id"],
        how="left"
    )
)
dataset.display()

In [0]:
# Define a window to handle multiple completions per received offer
completion_window = Window.partitionBy(
    "account_id", "offer_id", "t_received"
).orderBy("t_completed")

dataset = (
    dataset
    .withColumn(
        "t_completed_rank",
        F.row_number().over(completion_window)
    )
    .filter(
        (F.col("t_completed_rank") == 1) | F.col("t_completed").isNull()
    )
    .drop("t_completed_rank")
)
dataset.display()

In [0]:
# An offer is considered converted if it is completed within its validity period
dataset = dataset.withColumn(
    "converted",
    F.when(
        (F.col("t_completed").isNotNull()) &
        (F.col("t_completed") <= F.col("t_received") + F.col("duration")),
        F.lit(1)
    ).otherwise(F.lit(0))
)
dataset.display()

In [0]:
# Check class balance
dataset.groupBy("converted").count().show()

# Ensure 1 row = 1 customer–offer exposure
dataset.select(
    "account_id", "offer_id", "t_received"
).distinct().count(), dataset.count()

In [0]:
plot_categorical_distribution_same_style(dataset, ["converted"])

In [0]:
spark_shape(dataset)

### 5.2) Add features

#### 5.2.1) Offers' features

In [0]:
offers_final.display()

In [0]:
# Select offer-level features known at send time
offer_features = offers_final.select(
    F.col("id").alias("offer_id"),
    "offer_type",
    "discount_value",
    #"duration",
    "min_value",
    "channel_mobile",
    "channel_email",
    "channel_social",
    "channel_web"
)

# Join offer features to the modeling dataset
dataset_feat = (
    dataset
    .join(offer_features, on="offer_id", how="left")
)
dataset_feat.display()

#### 5.2.2) Profiles' features

In [0]:
profiles_final.display(profiles_final)

In [0]:
cols_to_drop = [
    "registered_on",
    "registered_on_date",
    "gender_index"
]
profiles_final = profiles_final.drop(*cols_to_drop)
profiles_final = profiles_final.withColumn(
    "cohort_month",
    F.month(F.to_date(F.col("cohort"), "yyyy-MM"))
)
profiles_final.display()

In [0]:
# Select customer profile features
profile_features = profiles_final.select(
    F.col("id").alias("account_id"),
    "age",
    "gender",
    "credit_card_limit",
    "cohort"   # YYYY-MM derived earlier
)

# Join customer features
dataset_feat = (
    dataset_feat
    .join(profile_features, on="account_id", how="left")
)
dataset_feat.display()

#### 5.2.3) Transactions' features

In [0]:
transactions_final.display()

In [0]:
# Keep only relevant columns from transactions
tx = transactions_final.select(
    "account_id",
    "event",
    "amount",
    "reward",
    "time_since_test_start"
)

In [0]:
dataset.display()

In [0]:
ds_ref = dataset.select("account_id", "offer_id", "t_received") #use table that has the exposition info of the offer (cusomer X received the offer Y at time T)

tx_with_ref = (
    tx.join(
        ds_ref,
        on="account_id",   # join ONLY by customer to check the behavior of the customer before receiving an offer
        how="inner"
    )
)


In [0]:
tx_pre_offer = tx_with_ref.filter(
    F.col("time_since_test_start") < F.col("t_received")
)
# t_received is the time when the offer was received
#keep features before the offer was received to avoid data leakage

In [0]:
tx_features = (
    tx_pre_offer
    .groupBy("account_id", "offer_id", "t_received")
    .agg(
        # How active is this customer before the offer?
        F.count("*").alias("num_events_before"),

        # How many transactions (events == 'transaction')?
        F.sum(
            F.when(F.col("event") == "transaction", 1).otherwise(0)
        ).alias("num_transactions_before"),

        # Total money spent before this offer
        F.sum("amount").alias("total_amount_before"),

        # On how many distinct days did the customer have activity?
        F.countDistinct("time_since_test_start").alias("active_days_before"),

        # Last time the customer did anything before this offer
        F.max("time_since_test_start").alias("last_event_time_before")
    )
)


In [0]:
tx_features.printSchema()
tx_features.show(5)

In [0]:
tx_features = (
    tx_features
    .withColumn(
        "recency",
        F.col("t_received") - F.col("last_event_time_before")
    )
    .drop("last_event_time_before")
)


In [0]:
dataset_feat = (
    dataset_feat
    .join(
        tx_features,
        on=["account_id", "offer_id", "t_received"],
        how="left"
    )
)

In [0]:
dataset_feat.printSchema()

In [0]:
dataset_feat.display()

In [0]:
dataset_feat.printSchema()

#### 5.2.4) Sanity checks

In [0]:
# Check final schema
dataset_feat.printSchema()

# Check row count consistency
print("Rows before features:", dataset.count())
print("Rows after features:", dataset_feat.count())

# Check target balance
dataset_feat.groupBy("converted").count().show()

### 5.3) Save dataset

In [0]:
#dataset_feat.write \
#    .mode("overwrite") \
#    .parquet("/Volumes/workspace/default/processed")

In [0]:
#dataset_feat_read = spark.read.parquet("/Volumes/workspace/default/processed")
#dataset_feat_read.display()